# ZeroER++ — block-filter full-data benchmarks (phase 2)

Takes each dataset's `<dataset>_block-filter` winner — best F1, tie-broken toward the smallest candidate set (`utils.front_trials`, delta 0.005) — and re-evaluates it on the **full data**, into sibling `<dataset>_block-filter_full` studies in the same `outputs/optuna.db`. Run `hyperparameter-tuning.ipynb` first.

## Setup

Generic runners/views in `utils`; the objective + its knobs are declared below and **must match the tuning run** (enqueued configs replay through the same distributions).

In [ ]:
from pyjedai_module import ZeroEREstimator
from utils import (DATASETS, DEFAULT_TUNE_FRACTION, load_data,
                   embedding_models, load_embeddings,       # embeddings helpers
                   precision_recall_f1, blocking_recall,    # objective helpers
                   retrain_all, export, show_results)

## Search space

Same `SPACE`/`SEED` as tuning; no budget needed — phase 2 only enqueues the existing best trials.

In [ ]:
SEED = 0
EXPERIMENT = "block-filter"   # study names: <dataset>_block-filter[_full]
SPACE = {                     # blocking/filtering knobs only — the matcher stays stock ZeroER
    # "embedding_model" is added per dataset in the objective: every model
    # discovered under datasets/embeddings/<dataset>/ is a candidate.
    "top_k": (1, 10),                          # neighbours kept per entity
    "similarity_distance": ["cosine", "euclidean"],
}

## Objective

The per-experiment trial — declared here, not in `utils`, so a new experiment is just a copy-and-tweak of this cell. It returns **F1 only** and carries its own `directions`, `experiment` (the study-name suffix) and `seed_trials` (per-model coverage guarantee); `make_objective` binds it to `SPACE`/`SEED` for the runners.

In [ ]:
class BlockFilterObjective:
    """Single-objective trial: **max F1**, searching only blocking & filtering —
    k-NN candidate generation over precomputed LLM record embeddings. The
    ZeroER matcher runs stock (default ``c_bay``/``max_iter``), so any F1 gain
    is attributable to the blocking stage alone.

    One ``Data`` is loaded per instance and reused across trials; embeddings
    are re-read per trial (mmap + row-select, cheap). ``get_params`` *is* the
    search space (keys map one-to-one onto the ZeroEREstimator ctor);
    ``seed_trials`` guarantees every embedding model is visited at least once
    before TPE takes over.
    """
    directions = ["maximize"]                 # F1 only
    experiment = EXPERIMENT                   # -> study "<dataset>_block-filter"

    def __init__(self, dataset, space, seed=0, partition=True):
        self.dataset = dataset
        self.source_trial = None              # set by retrain_full to stamp each full trial
        self.data = load_data(dataset, partition=partition, seed=seed)
        self.attributes = DATASETS[dataset]["attributes"]
        models = embedding_models(dataset)
        if not models:
            raise FileNotFoundError(
                f"no precomputed embeddings under datasets/embeddings/{dataset}/")
        self.space = dict(space, embedding_model=models)   # per-dataset model list
        frac = DATASETS[dataset].get("tune_fraction", DEFAULT_TUNE_FRACTION)
        scope = f"partition (frac={frac})" if partition else "full data"
        print(f"    {scope}: {self.data.num_of_entities_1}+{self.data.num_of_entities_2}"
              f" entities, {len(self.data.ground_truth)} matches, "
              f"{len(models)} embedding models", flush=True)

    def seed_trials(self):
        # one enqueued config per model (median top_k, cosine) — "go through
        # all the models" holds even under a sampled, non-grid budget
        mid_k = sum(self.space["top_k"]) // 2
        return [dict(embedding_model=m, top_k=mid_k, similarity_distance="cosine")
                for m in self.space["embedding_model"]]

    def get_params(self, trial):
        s = self.space
        return dict(
            embedding_model=trial.suggest_categorical("embedding_model", s["embedding_model"]),
            top_k=trial.suggest_int("top_k", *s["top_k"]),
            similarity_distance=trial.suggest_categorical("similarity_distance", s["similarity_distance"]),
        )

    def __call__(self, trial):
        params = self.get_params(trial)
        vectors = load_embeddings(self.dataset, params.pop("embedding_model"), self.data)
        est = ZeroEREstimator(blocker="precomputed", vectors=vectors,
                              attributes=self.attributes, **params)
        graph = est.fit_predict(self.data)
        _, _, f1 = precision_recall_f1(graph, self.data)
        trial.set_user_attr("candset_size", est.candset_size)
        trial.set_user_attr("pair_completeness",
                            round(blocking_recall(est.blocks, self.data), 4))
        trial.set_user_attr("blocking_seconds", round(est.blocking_seconds, 3))
        trial.set_user_attr("em_seconds", round(est.matcher.em_time, 3))
        if self.source_trial is not None:
            trial.set_user_attr("source_trial", self.source_trial)
        return f1


def make_objective(dataset, partition=True):                 # factory the runners call per dataset
    return BlockFilterObjective(dataset, SPACE, SEED, partition=partition)

## Run phase 2 — retrain each front on the full data

Resumable: configs already retrained (stamped `source_trial`) are skipped.

In [ ]:
retrain_all(make_objective, seed=SEED)

## Results

This notebook produces the `full` studies; the `partition` ones are shown too when present in `optuna.db`. `export` regenerates the CSV views (`*_block-filter*.csv`) from the db.

In [ ]:
show_results(experiment=EXPERIMENT)

In [ ]:
export(experiment=EXPERIMENT)

In [ ]:
import sqlite3

conn = sqlite3.connect("./outputs/optuna.db")
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cur.fetchall())

In [ ]:
cur.execute("SELECT * FROM studies;")
print(cur.fetchall())